# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [56]:
import pandas as pd
import numpy as np

---
## 2. Load data

In [3]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [4]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [5]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [6]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
#diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,price_date,dec_1,dec_2,price_off_peak_var,price_off_peak_fix,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,2015-12-01,0.119906,40.728885,0.126098,40.565969,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,2015-12-01,0.143943,44.444710,0.148047,44.266931,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,2015-12-01,0.201280,45.944710,0.150837,44.444710,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,2015-12-01,0.113068,40.728885,0.123086,40.565969,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,2015-12-01,0.145440,44.266930,0.149434,44.266931,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [7]:
diff['energy_pct_change'] = diff['offpeak_diff_dec_january_energy'] / diff['price_off_peak_var']
diff['power_pct_change'] = diff['offpeak_diff_dec_january_power']/diff['price_off_peak_fix']
diff.head()

,id,price_date,dec_1,dec_2,price_off_peak_var,price_off_peak_fix,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,energy_pct_change,power_pct_change
0,0002203ffbb812588b632b9e628cc38d,2015-12-01,0.119906,40.728885,0.126098,40.565969,-0.006192,0.162916,-0.049105,4.016066e-03
1,0004351ebdd665e6ee664792efc4fd13,2015-12-01,0.143943,44.444710,0.148047,44.266931,-0.004104,0.177779,-0.027721,4.016063e-03
2,0010bcc39e42b3c2131ed2ce55246e3c,2015-12-01,0.201280,45.944710,0.150837,44.444710,0.050443,1.500000,0.334421,3.374980e-02
3,0010ee3855fdea87602a5b7aba8e42de,2015-12-01,0.113068,40.728885,0.123086,40.565969,-0.010018,0.162916,-0.081390,4.016066e-03
4,00114d74e963e47177db89bc70108537,2015-12-01,0.145440,44.266930,0.149434,44.266931,-0.003994,-0.000001,-0.026728,-2.710827e-08


energy prices moved a lot more in relative terms (up to +33%, down to -8%) than power prices (mostly under 1%, one customer at 3%). If price is behind the churn, the energy-price column is probably the stronger candidate,

In [8]:
df = pd.merge(df, diff[['id', 'energy_pct_change', 'power_pct_change']], on='id', how='left')
df.shape

(14606, 46)

In [9]:
df[['energy_pct_change', 'power_pct_change']].isnull().sum()

energy_pct_change     20
power_pct_change     105
dtype: int64

In [10]:
df[['churn', 'energy_pct_change']].head()

,churn,energy_pct_change
0,1,0.159213
1,0,-0.024887
2,0,-0.027077
3,0,-0.029693
4,0,-0.049105


In [13]:
df[df['id'].isin(price_df)]

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,energy_pct_change,power_pct_change


In [26]:
check = diff[diff['id'].isin(nf['id'])]
check['price_off_peak_fix'].describe()

count    105.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: price_off_peak_fix, dtype: float64

In [27]:
df['power_pct_change'] = df['power_pct_change'].fillna(0)
df[['energy_pct_change', 'power_pct_change']].isnull().sum()

energy_pct_change    20
power_pct_change      0
dtype: int64

In [28]:
nf2 = df[df['energy_pct_change'].isnull()]
nf2['id'].isin(price_df['id']).sum()

np.int64(20)

In [29]:
check2 = diff[diff['id'].isin(nf2['id'])]
check2[['price_off_peak_var', 'dec_1']].describe()

,price_off_peak_var,dec_1
count,20.0,20.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


In [30]:
df['energy_pct_change'] = df['energy_pct_change'].fillna(0)
df[['energy_pct_change', 'power_pct_change']].isnull().sum()

energy_pct_change    0
power_pct_change     0
dtype: int64

In [32]:
df.nunique()

id                                14606
channel_sales                         8
cons_12m                          11065
cons_gas_12m                       2112
cons_last_month                    4751
date_activ                         1796
date_end                            368
date_modif_prod                    2129
date_renewal                        386
forecast_cons_12m                 13993
forecast_cons_year                 4218
forecast_discount_energy             12
forecast_meter_rent_12m            3528
forecast_price_energy_off_peak      516
forecast_price_energy_peak          329
forecast_price_pow_off_peak          41
has_gas                               2
imp_cons                           7752
margin_gross_pow_ele               2391
margin_net_pow_ele                 2391
nb_prod_act                          10
net_margin                        11965
num_years_antig                      13
origin_up                             6
pow_max                             698


In [34]:
df[['margin_gross_pow_ele', 'margin_net_pow_ele']].corr()

,margin_gross_pow_ele,margin_net_pow_ele
margin_gross_pow_ele,1.000000,0.999914
margin_net_pow_ele,0.999914,1.000000


In [36]:
df = df.drop(columns=['margin_net_pow_ele'])

dropped the column because of the ~1 correlation, most values are duplicated so its best to remove unwanted columns

In [35]:
df[['energy_pct_change', 'var_year_price_off_peak_var']].corr()

,energy_pct_change,var_year_price_off_peak_var
energy_pct_change,1.000000,0.213142
var_year_price_off_peak_var,0.213142,1.000000


In [37]:
df['tenure_days'] = (pd.Timestamp('2016-01-01') - df['date_activ']).dt.days
df[['date_activ', 'tenure_days']].head()

,date_activ,tenure_days
0,2013-06-15,930
1,2009-08-21,2324
2,2010-04-16,2086
3,2010-03-30,2103
4,2010-01-13,2179


In [46]:
df['days_to_end'] = (df['date_end'] - pd.Timestamp('2016-01-01')).dt.days
df[['date_end', 'days_to_end']].head()

,date_end,days_to_end
0,2016-06-15,166
1,2016-08-30,242
2,2016-04-16,106
3,2016-03-30,89
4,2016-03-07,66


In [48]:
df['month_activ'] = df['date_activ'].dt.month

Made 4 features:

energy_pct_change and power_pct_change (relative price changes)
tenure_days
days_to_end
month_activ

In [49]:
df['has_gas'] = df['has_gas'].map({'t': 1, 'f': 0})

In [50]:
df = pd.get_dummies(df, columns=['channel_sales'], dtype=int)
df.columns

Index(['id', 'cons_12m', 'cons_gas_12m', 'cons_last_month', 'date_activ',
       'date_end', 'date_modif_prod', 'date_renewal', 'forecast_cons_12m',
       'forecast_cons_year', 'forecast_discount_energy',
       'forecast_meter_rent_12m', 'forecast_price_energy_off_peak',
       'forecast_price_energy_peak', 'forecast_price_pow_off_peak', 'has_gas',
       'imp_cons', 'margin_gross_pow_ele', 'nb_prod_act', 'net_margin',
       'num_years_antig', 'origin_up', 'pow_max',
       'var_year_price_off_peak_var', 'var_year_price_peak_var',
       'var_year_price_mid_peak_var', 'var_year_price_off_peak_fix',
       'var_year_price_peak_fix', 'var_year_price_mid_peak_fix',
       'var_year_price_off_peak', 'var_year_price_peak',
       'var_year_price_mid_peak', 'var_6m_price_off_peak_var',
       'var_6m_price_peak_var', 'var_6m_price_mid_peak_var',
       'var_6m_price_off_peak_fix', 'var_6m_price_peak_fix',
       'var_6m_price_mid_peak_fix', 'var_6m_price_off_peak',
       'var_6m_price_pe

In [51]:
df = pd.get_dummies(df, columns=['origin_up'], dtype=int)

In [52]:
df = df.drop(columns=['date_activ', 'date_end', 'date_modif_prod', 'date_renewal'])

In [53]:
df.select_dtypes(exclude='number').columns

Index(['id'], dtype='str')

In [57]:
df['cons_12m'] = np.log10(df['cons_12m'] + 1)
df['cons_12m'].describe()

count    14606.000000
mean         4.223939
std          0.884515
min          0.000000
25%          3.754023
50%          4.149727
75%          4.610285
max          6.792889
Name: cons_12m, dtype: float64

In [58]:
skewed_cols = [
    'cons_gas_12m', 'cons_last_month', 'forecast_cons_12m',
    'forecast_cons_year', 'forecast_discount_energy',
    'forecast_meter_rent_12m', 'forecast_price_energy_off_peak',
    'forecast_price_energy_peak', 'forecast_price_pow_off_peak'
]

df[skewed_cols].min()

cons_gas_12m                      0.0
cons_last_month                   0.0
forecast_cons_12m                 0.0
forecast_cons_year                0.0
forecast_discount_energy          0.0
forecast_meter_rent_12m           0.0
forecast_price_energy_off_peak    0.0
forecast_price_energy_peak        0.0
forecast_price_pow_off_peak       0.0
dtype: float64

In [59]:
df[skewed_cols] = np.log10(df[skewed_cols] + 1)
df[skewed_cols].describe()

,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak
count,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000
mean,0.779244,2.264646,2.962177,1.784610,0.050918,1.517203,0.055766,0.020918,1.636058
std,1.717071,1.769305,0.683592,1.584986,0.267388,0.571481,0.009438,0.020296,0.134237
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,2.695477,0.000000,0.000000,1.235023,0.047796,0.000000,1.619163
50%,0.000000,2.899547,3.046836,2.498311,0.000000,1.296555,0.058109,0.035085,1.656207
75%,0.000000,3.529430,3.380716,3.242231,0.000000,2.120673,0.059316,0.040933,1.656207
max,6.618528,5.887169,4.918575,5.243970,1.491362,2.778376,0.105157,0.077722,1.780075


dropped margin_net_pow_ele (99.99% correlated with the gross margin), created the price-change percentages, filled the 0 ÷ 0 blanks with 0 because the price hadn't changed, created tenure_days, days_to_end and month_activ, converted has_gas, channel_sales and origin_up to numbers, and log-transformed the skewed columns